# Limpieza de Datos

## Inspeción preliminar

In [35]:
import pandas as pd
import pathlib
import re
import unicodedata
import ast

In [ ]:
# Ruta recomendada para compartir el proyecto sin versionar datos sensibles.
path_todos_registros = pathlib.Path("../data_raw/todos_registros.xlsx")

# Fallback para el entorno local original del proyecto.
if not path_todos_registros.exists():
    path_todos_registros = pathlib.Path(r"D:\Open_Vzla_SOS_Calidad_de_Datos\aevscraping\datos_consolidados\todos_registros.xlsx")

df_raw = pd.read_excel(path_todos_registros)

# Conservamos el scraping original intacto y trabajamos siempre sobre una copia.
df_clean = df_raw.copy()
df_clean.head()


In [37]:
# Filas y columnas del DataFrame
df_clean.shape

(241463, 16)

In [38]:
# Información general del DataFrame
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 241463 entries, 0 to 241462
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   ID                    241463 non-null  int64  
 1   Nombre                241463 non-null  str    
 2   Cédula                241463 non-null  str    
 3   Edad                  140396 non-null  float64
 4   Última Ubicación      228992 non-null  str    
 5   Teléfono Contacto     101095 non-null  str    
 6   Observaciones         162535 non-null  str    
 7   Estado                241463 non-null  str    
 8   Ubicación Encontrado  14678 non-null   str    
 9   Encontrado Por        16091 non-null   str    
 10  Cédula Encontrado     0 non-null       float64
 11  URL Foto              129962 non-null  str    
 12  Fecha Registro        241463 non-null  str    
 13  Fecha Actualización   241463 non-null  str    
 14  Es Menor              241463 non-null  str    
 15  Fuente     

### Inspección de los datos cargados en la columna Cédula

In [39]:
cedulas_no_numericas = df_clean[
    ~df_clean["Cédula"].astype(str).str.fullmatch(r"\d+")
]["Cédula"].value_counts(dropna=False)

cedulas_no_numericas

Cédula
N/D              201685
No registrado     27942
Name: count, dtype: int64

In [40]:
df_clean["Cédula"].astype(str).str.fullmatch(r"\d+").value_counts()

Cédula
False    229627
True      11836
Name: count, dtype: int64

In [41]:
df_clean["longitud_cedula"] = df_clean["Cédula"].astype(str).str.len()

df_clean["longitud_cedula"].value_counts().sort_index()

longitud_cedula
1          6
2         32
3     201699
4         14
5          5
6         38
7       2631
8       8065
9        834
10       135
11        18
12         7
13     27943
14         4
15         6
16        23
20         2
24         1
Name: count, dtype: int64

In [42]:
df_clean.loc[
    df_clean["longitud_cedula"] == 6,
    "Cédula"
].value_counts().head(20)

Cédula
811810    1
646450    1
179657    1
186322    1
100000    1
514319    1
925861    1
584253    1
069958    1
240759    1
123456    1
460882    1
948812    1
122222    1
993796    1
949052    1
933421    1
507176    1
111111    1
123452    1
Name: count, dtype: int64

In [43]:
df_clean.loc[
    df_clean["longitud_cedula"] == 9,
    "Cédula"
].value_counts().head(20)

Cédula
152804440    1
284400240    1
323598830    1
119936420    1
126606800    1
374549870    1
203035080    1
343169290    1
488153620    1
171163730    1
279047940    1
189105000    1
108107460    1
192737600    1
251329460    1
296653000    1
188095590    1
163099760    1
236915550    1
320743330    1
Name: count, dtype: int64

In [44]:
df_clean.loc[
    df_clean["longitud_cedula"].isin([11, 12]),
    ["Nombre", "Cédula", "Teléfono Contacto", "Fuente"]
]

,Nombre,Cédula,Teléfono Contacto,Fuente
243,Cadena Franyerson,41259757250,0212-5741111,venezuelatebusca
333,Moises Nieves,42415357070,0212-5741111,venezuelatebusca
11566,Danny Herrera,04242457733,04128024657,NaN
13091,Lurrise Gil,173783937382,+58 424-1629687,NaN
13230,Christopher Camero,04124443610,04124443610,NaN
13601,Jesús Guerrero,04162009040,04162009040,NaN
13645,Yoel Salazar,584248598707,+5595991670665,NaN
14580,Yaraima Ruiz de Innaty,04149023294,+104149023294,NaN
15164,Alahim Rodriguez y Familia Cubana Yadina Yanez...,04144314297,04144314297,NaN
15206,Eliana Mogollon,04127744096,04127744096,NaN


In [45]:
df_clean.loc[
    df_clean["longitud_cedula"] == 8,
    "Cédula"
].value_counts().head(20)

Cédula
17307478    2
37425922    2
37099131    1
31533678    1
31201015    1
31712959    1
31160449    1
15507716    1
11072271    1
21509418    1
10711859    1
13499932    1
17424256    1
19627731    1
36468562    1
16952514    1
25220034    1
29933537    1
34054842    1
79907280    1
Name: count, dtype: int64

In [46]:
# Sanitización cédulas: Solo 7 u 8 dígitos y excluir valores donde todos los dígitos son iguales
cedulas = df_clean["Cédula"].astype(str)

# Solo 7 u 8 dígitos
mask_longitud = cedulas.str.fullmatch(r"\d{7,8}")

# Excluir valores donde todos los dígitos son iguales
mask_repetidos = cedulas.apply(lambda x: len(set(x)) == 1 if x.isdigit() else False)

df_clean["Cédula_clean"] = pd.NA
df_clean.loc[mask_longitud & ~mask_repetidos, "Cédula_clean"] = cedulas

# Inspección de los datos cargados en la columna Nombre

In [47]:
df_clean["Nombre"].duplicated().sum()

np.int64(151984)

In [48]:
df_clean[df_clean["Nombre"].duplicated(keep=False)].sort_values("Nombre")[["Nombre", "Cédula", "Fuente", "URL Foto"]]

,Nombre,Cédula,Fuente,URL Foto
110240,"""><svg/onload=(""@jofpin"");>",N/D,redayudavenezuela,NaN
202241,"""><svg/onload=(""@jofpin"");>",N/D,desaparecidosterremoto,NaN
167863,(Gabriel) Nacho Marín,N/D,redayudavenezuela,https://terremotovenezuela.app/api/missing/634...
191988,(Gabriel) Nacho Marín,N/D,desaparecidosterremoto,https://cdn-imagenes.theempire.tech/images/pf3...
172165,"(rojo) Wiston, (negro) andres",N/D,redayudavenezuela,https://reconexion-api-images-147455119818.s3....
...,...,...,...,...
163558,🙏,N/D,redayudavenezuela,https://terremotovenezuela.app/api/missing/eef...
163559,🙏,N/D,redayudavenezuela,https://terremotovenezuela.app/api/missing/eef...
163560,🙏,N/D,redayudavenezuela,https://terremotovenezuela.app/api/missing/eef...
163561,🙏,N/D,redayudavenezuela,https://terremotovenezuela.app/api/missing/eef...


In [49]:
# Sanitización de nombres: eliminar HTML, XSS, caracteres raros y dejar solo letras, espacios y acentos
def limpiar_nombre(texto):
    if pd.isna(texto):
        return pd.NA

    texto = str(texto)

    # 1. eliminar HTML / tags tipo SVG / script
    texto = re.sub(r"<.*?>", "", texto)

    # 2. eliminar patrones tipo XSS (onload, javascript, etc.)
    texto = re.sub(r"(?i)(on\w+=|javascript:|script)", "", texto)

    # 3. normalizar unicode (quita caracteres raros)
    texto = unicodedata.normalize("NFKD", texto)

    # 4. dejar solo letras, espacios y acentos
    texto = re.sub(r"[^a-zA-ZÀ-ÿñÑ\s]", "", texto)

    # 5. limpiar espacios múltiples
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto if texto != "" else pd.NA


df_clean["Nombre_clean"] = df_clean["Nombre"].apply(limpiar_nombre)

In [50]:
# Filtrar nombres válidos 
df_clean[df_clean["Nombre_clean"].notna()].head()

,ID,Nombre,Cédula,Edad,Última Ubicación,Teléfono Contacto,Observaciones,Estado,Ubicación Encontrado,Encontrado Por,Cédula Encontrado,URL Foto,Fecha Registro,Fecha Actualización,Es Menor,Fuente,longitud_cedula,Cédula_clean,Nombre_clean
0,7410076847799689,José Gregorio Rojas Bello,No registrado,NaN,La Guaira,+584142504728,No sabemos d el y su familia padre esposa hijos,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/7ec6...,2026-06-28T17:50:42.114Z,2026-06-28T17:50:42.114Z,No,venezuelatebusca,13,<NA>,Jose Gregorio Rojas Bello
1,2590484473230077,Hector Reyes,No registrado,NaN,San Agustin,+584124989844,Acabo de ver un vídeo de WhatsApp donde tienen...,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/cc73...,2026-06-28T17:50:34.923Z,2026-06-28T17:50:34.923Z,No,venezuelatebusca,13,<NA>,Hector Reyes
2,152612323455194,Neiderlyn Alfonso,No registrado,6.0,Tanaguarenas la Guaira los cocos,+584241592570,NaN,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/ba9b...,2026-06-28T17:49:37.868Z,2026-06-28T17:49:37.868Z,Sí,venezuelatebusca,13,<NA>,Neiderlyn Alfonso
3,2394053969385436,Samuel Vicente Cortez Olivier,37099131,11.0,"Conjunto Residencial Caribe, la Guaira",+584127387227,Pantalon azul franela,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/53e6...,2026-06-28T17:48:07.144Z,2026-06-28T17:48:07.144Z,Sí,venezuelatebusca,8,37099131,Samuel Vicente Cortez Olivier
4,6937486286932653,Amir Infante Galván,No registrado,16.0,La guaira,+584162443424,NaN,Desaparecido,NaN,NaN,NaN,NaN,2026-06-28T17:48:04.530Z,2026-06-28T17:48:04.530Z,Sí,venezuelatebusca,13,<NA>,Amir Infante Galvan


In [51]:
df_clean["Nombre_clean"].value_counts().head(20)

Nombre_clean
ll                    29625
J                     19348
a                     14942
NN                    10127
d                      2639
L                      1857
Yo                     1729
F                       221
Clementina Segovia       24
Zulay Sarmiento          23
Vicky Urdaneta           22
Osmary Montilla          21
Carlos Gonzalez          21
Alicia Falcon            21
Gustavo Ruiz             21
Nancy Hernandez          21
Luis Auyanet             20
Camila Gutierrez         20
Eliana Mogollon          19
Orlando Gonzalez         19
Name: count, dtype: int64

In [52]:
# Filtro básico de calidad de nombre
df_clean = df_clean[
    df_clean["Nombre_clean"].notna() &
    (df_clean["Nombre_clean"].str.len() > 3)
]

In [53]:
# Eliminar patrones basura
basura = {"nn", "na", "ll", "j", "a", "d", "l", "yo"}

df_clean = df_clean[
    ~df_clean["Nombre_clean"].str.lower().isin(basura)
]

In [54]:
# Detectar “parece nombre real”
df_clean = df_clean[
    df_clean["Nombre_clean"].str.contains(r"^[A-Za-zÁÉÍÓÚÑáéíóúñ ]{4,}$", regex=True)
]

In [55]:
df_clean["Nombre_clean"]

0                Jose Gregorio Rojas Bello
1                             Hector Reyes
2                        Neiderlyn Alfonso
3            Samuel Vicente Cortez Olivier
4                      Amir Infante Galvan
                        ...               
241458                          Maria mata
241459                   rodrigo rodriguez
241460    Skarlent Rodriguez y Jose Castro
241461                        Jesus sabala
241462                    Liliagnit Borges
Name: Nombre_clean, Length: 157700, dtype: str

In [56]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 157700 entries, 0 to 241462
Data columns (total 19 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   ID                    157700 non-null  int64  
 1   Nombre                157700 non-null  str    
 2   Cédula                157700 non-null  str    
 3   Edad                  121648 non-null  float64
 4   Última Ubicación      145230 non-null  str    
 5   Teléfono Contacto     98630 non-null   str    
 6   Observaciones         98601 non-null   str    
 7   Estado                157700 non-null  str    
 8   Ubicación Encontrado  14674 non-null   str    
 9   Encontrado Por        16087 non-null   str    
 10  Cédula Encontrado     0 non-null       float64
 11  URL Foto              113057 non-null  str    
 12  Fecha Registro        157700 non-null  str    
 13  Fecha Actualización   157700 non-null  str    
 14  Es Menor              157700 non-null  str    
 15  Fuente          

In [57]:
# Duplicados exactos (fila completa)
df_clean.duplicated().sum()

np.int64(0)

In [58]:
# Duplicados por Cédula
df_clean["Cédula_clean"].duplicated().sum()

np.int64(147014)

In [59]:
df_clean[df_clean["Cédula"].duplicated(keep=False)].sort_values("Cédula")

,ID,Nombre,Cédula,Edad,Última Ubicación,Teléfono Contacto,Observaciones,Estado,Ubicación Encontrado,Encontrado Por,Cédula Encontrado,URL Foto,Fecha Registro,Fecha Actualización,Es Menor,Fuente,longitud_cedula,Cédula_clean,Nombre_clean
2738,5214567643934092,María Laura Hernández Acosta,17307478,42.0,Playa grande la guaira | Hotel Catimar,+34697293314,"No sé sabe | Mujer de 1,70 con mechas. \nUtili...",Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/5944...,2026-06-28T02:19:45.262Z,2026-06-28T12:11:22.592Z,No,venezuelatebusca,8,17307478,Maria Laura Hernandez Acosta
17161,6308673208560697,Maria Laura Hernandez Acosta,17307478,42.0,Hotel Catimar,+584145198367,"Mujer de 1,70 con mechas. \n\nUtiliza reloj ne...",Desaparecido,NaN,NaN,NaN,NaN,2026-06-25T22:17:37.075Z,2026-06-26T11:25:48.999Z,No,NaN,8,17307478,Maria Laura Hernandez Acosta
3168,3271040832349847,Leslie Zaragoza,37425922,10.0,La guaira los corales vivienda vene8,+584165220398,Catirita ojos ámbar delgadita cabello color ch...,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/166a...,2026-06-27T19:57:41.877Z,2026-06-28T02:00:30.978Z,Sí,venezuelatebusca,8,37425922,Leslie Zaragoza
8426,8705332788312286,Leslie Zaragoza,37425922,10.0,La Guaira - caribe,04128983063,NaN,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/52fa...,2026-06-26T17:50:00.779Z,2026-06-26T17:50:00.779Z,Sí,NaN,8,37425922,Leslie Zaragoza
2418,7758004926016465,Carmen Josefa Guzmán Palacios,5491667,68.0,Edificio tahiti o Tamiami Caraballeda | La gua...,+584241373304,"Mujer de 1.60 metros aproximadamente, gordita,...",Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/da8a...,2026-06-28T05:24:59.194Z,2026-06-28T12:11:22.592Z,No,venezuelatebusca,7,5491667,Carmen Josefa Guzman Palacios
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39773,8986422262758169,Sofia Valentina Delgado Brito,No registrado,15.0,La Caraballeda La Guaira,+584122974445,NaN,Desaparecido,NaN,NaN,NaN,NaN,2026-06-26T23:12:50.768Z,2026-06-26T23:12:50.768Z,Sí,NaN,13,<NA>,Sofia Valentina Delgado Brito
39774,8873460803865277,Lilian del Valle Vasquez,No registrado,40.0,Edificio Elite Bech,+5511913295290,NaN,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/b8ee...,2026-06-26T23:11:55.254Z,2026-06-26T23:11:55.254Z,No,NaN,13,<NA>,Lilian del Valle Vasquez
39775,1710186564796758,Jonathan Ortega,No registrado,39.0,La guaira,+5569992937643,NaN,Desaparecido,NaN,NaN,NaN,NaN,2026-06-26T23:09:51.660Z,2026-06-26T23:09:51.660Z,No,NaN,13,<NA>,Jonathan Ortega
39776,1278031129667196,José Ramon Espinoza,No registrado,NaN,"La Guaira, Los corales",+584241691539,NaN,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/79a3...,2026-06-26T23:09:41.367Z,2026-06-26T23:09:41.367Z,No,NaN,13,<NA>,Jose Ramon Espinoza


In [60]:
# Duplicados por NOMBRE + CÉDULA
df_clean.duplicated(subset=["Cédula_clean", "Nombre_clean"]).sum()

np.int64(65491)

In [61]:
df_clean[
    df_clean.duplicated(subset=["Cédula_clean", "Nombre_clean"], keep=False)
].sort_values(["Cédula_clean", "Nombre_clean"]).head(50)

,ID,Nombre,Cédula,Edad,Última Ubicación,Teléfono Contacto,Observaciones,Estado,Ubicación Encontrado,Encontrado Por,Cédula Encontrado,URL Foto,Fecha Registro,Fecha Actualización,Es Menor,Fuente,longitud_cedula,Cédula_clean,Nombre_clean
2738,5214567643934092,María Laura Hernández Acosta,17307478,42.0,Playa grande la guaira | Hotel Catimar,+34697293314,"No sé sabe | Mujer de 1,70 con mechas. \nUtili...",Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/5944...,2026-06-28T02:19:45.262Z,2026-06-28T12:11:22.592Z,No,venezuelatebusca,8,17307478,Maria Laura Hernandez Acosta
17161,6308673208560697,Maria Laura Hernandez Acosta,17307478,42.0,Hotel Catimar,+584145198367,"Mujer de 1,70 con mechas. \n\nUtiliza reloj ne...",Desaparecido,NaN,NaN,NaN,NaN,2026-06-25T22:17:37.075Z,2026-06-26T11:25:48.999Z,No,NaN,8,17307478,Maria Laura Hernandez Acosta
3168,3271040832349847,Leslie Zaragoza,37425922,10.0,La guaira los corales vivienda vene8,+584165220398,Catirita ojos ámbar delgadita cabello color ch...,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/166a...,2026-06-27T19:57:41.877Z,2026-06-28T02:00:30.978Z,Sí,venezuelatebusca,8,37425922,Leslie Zaragoza
8426,8705332788312286,Leslie Zaragoza,37425922,10.0,La Guaira - caribe,04128983063,NaN,Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/52fa...,2026-06-26T17:50:00.779Z,2026-06-26T17:50:00.779Z,Sí,NaN,8,37425922,Leslie Zaragoza
2418,7758004926016465,Carmen Josefa Guzmán Palacios,5491667,68.0,Edificio tahiti o Tamiami Caraballeda | La gua...,+584241373304,"Mujer de 1.60 metros aproximadamente, gordita,...",Desaparecido,NaN,NaN,NaN,https://venezuelatebusca.com/media/photos/da8a...,2026-06-28T05:24:59.194Z,2026-06-28T12:11:22.592Z,No,venezuelatebusca,7,5491667,Carmen Josefa Guzman Palacios
31856,5221351944823551,Carmen Josefa Guzmán Palacios,5491667,68.0,La guaira caraballeda,+525534762784,"Mujer de 1.60 metros aproximadamente, gordita,...",Desaparecido,NaN,NaN,NaN,NaN,2026-06-25T08:05:20.587Z,2026-06-26T11:25:48.999Z,No,NaN,7,5491667,Carmen Josefa Guzman Palacios
7184,2825446193185940,A. Baez Maria,No registrado,NaN,NaN,NaN,NaN,Localizado,NaN,NaN,NaN,NaN,2026-06-26T19:12:00.000Z,2026-06-26T21:45:00.466Z,No,NaN,13,<NA>,A Baez Maria
16318,8306590905748410,A Baez Maria,No registrado,NaN,NaN,NaN,NaN,Localizado,NaN,NaN,NaN,NaN,2026-06-25T22:56:46.819Z,2026-06-25T22:56:46.819Z,No,NaN,13,<NA>,A Baez Maria
169210,2669704492057698816,A Baez Maria,N/D,NaN,NaN,NaN,NaN,Localizado,NaN,NaN,NaN,NaN,2026-06-25T22:56:46.819000+00:00,2026-06-26T00:58:03.810545+00:00,No,redayudavenezuela,3,<NA>,A Baez Maria
16326,1224704662704984,A. Rico Gustavo,No registrado,NaN,NaN,NaN,NaN,Localizado,NaN,NaN,NaN,NaN,2026-06-25T22:56:20.530Z,2026-06-25T22:56:20.530Z,No,NaN,13,<NA>,A Rico Gustavo


## Priorización y consolidación de duplicados por nombre

La regla de negocio para reducir duplicados por nombre no exige que toda la evidencia esté en la misma fila.

Primero se elige una fila base con mayor evidencia útil:

1. persona localizada/encontrada;
2. cédula válida;
3. teléfono disponible;
4. mayor cantidad de campos informativos;
5. fecha de actualización más reciente.

Luego se completan los campos vacíos de esa fila base con datos válidos del resto de registros del mismo `Nombre_clean`.


In [62]:
# Diagnóstico antes de reducir duplicados por nombre
resumen_duplicados_nombre = pd.Series({
    "filas_antes": len(df_clean),
    "nombres_unicos_antes": df_clean["Nombre_clean"].nunique(),
    "filas_con_nombre_duplicado_antes": df_clean.duplicated(subset=["Nombre_clean"], keep=False).sum(),
})

resumen_duplicados_nombre


filas_antes                         157700
nombres_unicos_antes                 84778
filas_con_nombre_duplicado_antes    122645
dtype: int64

In [63]:
# Eliminamos duplicados exactos por la identidad limpia más fuerte disponible.
df_clean = df_clean.drop_duplicates(
    subset=["Nombre_clean", "Cédula_clean", "Teléfono Contacto", "URL Foto"],
    keep="first"
).copy()


In [64]:
# Convertimos fechas una sola vez para usarlas como criterio de desempate.
df_clean["Fecha Registro"] = pd.to_datetime(
    df_clean["Fecha Registro"],
    errors="coerce"
)

df_clean["Fecha Actualización"] = pd.to_datetime(
    df_clean["Fecha Actualización"],
    errors="coerce"
)

# La edad se normaliza antes del ranking para evitar comparar textos con números.
df_clean["Edad"] = pd.to_numeric(df_clean["Edad"], errors="coerce")


In [65]:
def tiene_dato_valido(serie):
    """Identifica valores realmente informativos, no marcadores como 'No registrado'."""
    valores_no_informativos = {
        "", "nan", "none", "no registrado", "no registrada", "sin registro",
        "sin datos", "n/a", "na", "null", "no aplica"
    }

    texto = serie.astype("string").str.strip().str.lower()
    return serie.notna() & ~texto.isin(valores_no_informativos)


telefono_normalizado = (
    df_clean["Teléfono Contacto"]
    .astype("string")
    .str.replace(r"\D+", "", regex=True)
)

estado_normalizado = df_clean["Estado"].astype("string").str.lower()

df_clean["esta_localizado"] = (
    tiene_dato_valido(df_clean["Ubicación Encontrado"]) |
    tiene_dato_valido(df_clean["Encontrado Por"]) |
    tiene_dato_valido(df_clean["Cédula Encontrado"]) |
    estado_normalizado.str.contains(r"localiz|encontr", regex=True, na=False)
)

df_clean["tiene_cedula"] = df_clean["Cédula_clean"].notna()
df_clean["tiene_telefono"] = telefono_normalizado.str.len().ge(7).fillna(False)


In [66]:
columnas_informativas = [
    "Cédula_clean",
    "Teléfono Contacto",
    "Edad",
    "Última Ubicación",
    "Observaciones",
    "Estado",
    "Ubicación Encontrado",
    "Encontrado Por",
    "Cédula Encontrado",
    "URL Foto",
    "Fuente",
]

df_clean["campos_informativos"] = sum(
    tiene_dato_valido(df_clean[col]).astype(int)
    for col in columnas_informativas
)

# El ranking es vectorizado: evita loops lentos y deja explícita la prioridad de negocio.
df_clean = df_clean.sort_values(
    by=[
        "Nombre_clean",
        "esta_localizado",
        "tiene_cedula",
        "tiene_telefono",
        "campos_informativos",
        "Fecha Actualización",
        "Fecha Registro",
    ],
    ascending=[True, False, False, False, False, False, False],
    na_position="last",
)


In [67]:
columnas_control = ["esta_localizado", "tiene_cedula", "tiene_telefono", "campos_informativos"]
columnas_a_consolidar = [
    columna for columna in df_clean.columns
    if columna not in ["Nombre_clean", *columnas_control]
]

df_consolidacion = df_clean.copy()

# Convertimos valores no informativos en nulos para que groupby().first()
# tome el primer dato útil disponible por columna dentro de cada nombre.
for columna in columnas_a_consolidar:
    df_consolidacion.loc[~tiene_dato_valido(df_consolidacion[columna]), columna] = pd.NA

# Como df_clean ya está ordenado por prioridad, first() conserva lo mejor disponible
# aunque los datos útiles estén distribuidos en distintas filas duplicadas.
df_clean = (
    df_consolidacion
    .groupby("Nombre_clean", sort=False, as_index=False)
    .first()
    .drop(columns=columnas_control)
    .reset_index(drop=True)
)


In [68]:
resumen_deduplicacion_nombre = pd.Series({
    "filas_despues": len(df_clean),
    "nombres_unicos_despues": df_clean["Nombre_clean"].nunique(),
    "filas_con_nombre_duplicado_despues": df_clean.duplicated(subset=["Nombre_clean"], keep=False).sum(),
})

pd.concat([resumen_duplicados_nombre, resumen_deduplicacion_nombre])


filas_antes                           157700
nombres_unicos_antes                   84778
filas_con_nombre_duplicado_antes      122645
filas_despues                          84778
nombres_unicos_despues                 84778
filas_con_nombre_duplicado_despues         0
dtype: int64

## Normalización final


In [69]:
# Normalizamos textos solo después de decidir qué registro conservar.
columnas_texto_titulo = [
    "Nombre_clean",
    "Última Ubicación",
    "Encontrado Por",
    "Ubicación Encontrado",
    "Estado",
    "Es Menor",
]

for columna in columnas_texto_titulo:
    df_clean[columna] = df_clean[columna].astype("string").str.strip().str.title()

df_clean["Observaciones"] = df_clean["Observaciones"].astype("string").str.strip()


In [70]:
# Validación rápida de calidad posterior a la limpieza.
validacion_final = pd.Series({
    "filas_finales": len(df_clean),
    "duplicados_por_nombre": df_clean.duplicated(subset=["Nombre_clean"]).sum(),
    "registros_con_cedula_limpia": df_clean["Cédula_clean"].notna().sum(),
    "registros_con_telefono": tiene_dato_valido(df_clean["Teléfono Contacto"]).sum(),
})

validacion_final


filas_finales                  84778
duplicados_por_nombre          12193
registros_con_cedula_limpia    10387
registros_con_telefono         72385
dtype: int64

## Exportación pendiente

La exportación a Excel todavía no se ejecuta en este notebook porque la limpieza sigue en revisión.

Cuando la lógica de calidad de datos esté validada, este paso deberá generar el archivo limpio para alimentar el pipeline posterior.
